---
title: Week 4.6, Group Project: Analaysis of a leachate dataset
subtitle: Landfill Leachate
author:
  - name: Timo Heimovaara
    affiliations: Delft University of Technology, department of Geoscience & Engineering
    orcid: 
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-01-30
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Analsysis of the (partial) chemical composition of landfill leachate

TODO: Introduction
1. Context of the dataset
    - Landfill waste body
    - Methanogenisis in the bulk of the wastebody (PCO2 and PCO2 are about 0.5)
    - Wastebody contains a wide mixture of (reactive) compounds
2. Type of data collected
3. Why this analysis is relevant
4. What type of questions students need to solve
5. Explain what we expect them to do...


In [47]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

%matplotlib widget
sns.set()

# Prepare a file to capture PyOrchestra output
capture_file = open("pyorchestra_output.log", "w")


# pyOrchestra is implemented in C++
# Save original stdout file descriptor
# original_stdout_fd = sys.stdout.fileno()

# Duplicate original stdout so we can restore it later
# saved_stdout_fd = os.dup(original_stdout_fd)



# We need to import some Orchestra files. We need to know the path layout on 
# the local machine:
def find_book_root(start: Path | None = None) -> Path:
    """
    Walk upward from `start` (or CWD) until a directory containing Jupyter Book
    marker files is found. Returns the path to the book root.
    Raises FileNotFoundError if no root is found.
    """
    config_any = {"_config.yml", "_config.yaml"}      # some projectrs use .yaml
    myst_any = {"myst.yml", "myst.yaml"}            # jupyter-book uses _toc.yml

    cur = Path(Path.cwd()).resolve()

    for parent in [cur, *cur.parents]:
        children = {f.name for f in parent.iterdir()} if parent.exists() else set()
        has_any_config = bool(config_any & children)
        has_any_myst = bool(myst_any & children)
        if has_any_config and has_any_myst:
            return parent

    raise FileNotFoundError(
        f"Could not find Jupyter Book root (no _config.y* and myst.y* found above {cur})"
    )


def path_from_book_root(*parts: str | Path) -> Path:
    root = find_book_root()
    p = (root.joinpath(*parts)).resolve()
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p

# In order for orchestra to run we need to change directory to the directory with the input file:


# Example usage:
# input_file = path_from_book_root("content", "week 02", "Orchestra_simulation", "chemistry1.inp")
# print("Input file:", input_file)

orchestra_path = path_from_book_root("content", "project_THe", "Orchestra_Project")
# print(orchestra_path)

## Preparation and preliminary investigation of data

TODO: Add a dataset with the leachate production data (cumulative!)

First Week:

1. Import the data
2. Get a quick over view of the content and the structure of the dataset
3. Understand how to plot the time series in the data set, save the figures to a file and create an overview report.
4. Need to calculate molar concentrations from mg/l values
5. Think about what questions you want to resolve with this data?
    - Saturation status of the samples as they are;
    - What were mostly likely conditions where the samples originated?
    - What will happen to the samples if the leachate would be discharged to a system at atmospheric conditions.
5. Prepare the interface to PyOrchestra, have a look at provided GUI of Orchestra and the corresponding Chemistry File


Second Week:
1. Use Orchestra to find out which minerals may be present.
    - please note that Orchestra is an equilibrium calculation


Please note: 
need install openpyxl:  mamba install -c conda-forge openpyxl


We import the data and select the sample with most measurements to do the first Orchestra calculation

In [48]:
# %% 1
# import the data set from the excel file.
# We will use the data from the leachate monitoring at PP-11N.

df_leachate = pd.read_excel('data/df_macros_PP-11N.xlsx')

# %% 2
# Check which dates have most parameters measured.
# We will use these parameters as our input for the Orchestra 
# calculation.

# For some dates, less parameters were measured. We can then choose to work 
# with the estimated values, using the time series.

# df_macro_counts = (
#     df_leachate.group_by('measpointname','date')
#     .agg(pl.count('cname').alias('macro_count'))
#     .sort('macro_count', descending=True)
# )

df_par_counts = (
    df_leachate.groupby(['measpointname', 'date'])['cname']
    .count()
    .reset_index(name='macro_count')
    .sort_values('macro_count', ascending=False)
)

df_par_counts
date_with_most_pars = df_par_counts.iloc[0]['date']

# %%
# We use the date with most parameters for our first analysis
sel_idx = df_leachate['date'] == date_with_most_pars

df_work = df_leachate[sel_idx].copy()

# Export df_work to an Excel file so that we can 
# have a quick access to the parameters in it
# for setting up the translation from mg/l to moles/l 
# for the Orchestra input.
# %%
df_work.to_excel('tmp/df_work_PP-11N.xlsx', index=False)

# %%
# Using the content from the file we now create a table
# with the parameters, their values and the conversion to moles/l.
# We will use this table to set up the translation from mg/l to moles/l
# for the Orchestra input.

# Conversion table contaings molar masses for the parameters 
# in the df_work dataframe and the corresponding parameter names
# in the Orchestra input file.

# componentname, molar mass (g/mol), Orchestra parameter name
conversion_table = {
    'Sulfaat (als SO4)': [96.06, 'SO4-2.tot'],
    'Sulfide': [32.07, 'S-2.tot'],
    'Natrium [Na]': [22.99, 'Na+.tot'],
    'Nikkel [Ni]': [58.69, 'Ni+2.tot'],
    'IJzer [Fe]': [55.85, 'Fe+2.tot'],
    'Zink [Zn]': [65.38, 'Zn+2.tot'],
    'Magnesium [Mg]': [24.31, 'Mg+2.tot'],
    'Calcium [Ca]': [40.08, 'Ca+2.tot'],
    'Ammonium (als NH4)': [18.04, 'NH4+.tot'],
    'Chloride': [35.45, 'Cl-.tot'],
    'Bicarbonaat': [61.02, 'HCO3-.tot'],
    'Fosfaat (als PO4)': [94.97, 'PO4-3.tot'],
    'Kalium [K]': [39.10, 'K+.tot'],
    'Silicium [Si]': [28.09, 'Si.tot'],
    'Mangaan [Mn]': [54.94, 'Mn+2.tot'],
    'Arseen [As]': [74.92, 'As.tot'],
    'Temperatuur': [1e-3, 'T'] # please note the factor 1e-3 which will be corrected for in the conversion to moles/l.
}

# We can now use this conversion table 
# to convert the values in the df_work dataframe
# from mg/l to moles/l and to add a new column with Orchestra parameter names.

df_work['val_mol_l'] = df_work.apply(
    lambda row: 
        (row['val_mgl'] * 1e-3) / conversion_table[row['cname']][0] 
        if row['cname'] in conversion_table else row['val_mgl'], axis=1)

df_work['orchestra_param'] = df_work.apply(
    lambda row: 
        conversion_table[row['cname']][1] 
        if row['cname'] in conversion_table else row['cname'], axis=1)


# select temperatures and add 273.15 to convert to K
sel_temp = df_work['orchestra_param'] == 'T'
df_work.loc[sel_temp, 'val_mol_l'] += 273.15

# %%
# Rewrite df_work to excel so we can copy the contents to the Orchestra input file.
# df_work.to_excel('tmp/df_work_PP-11N_converted.xlsx', index=False)
# %%


## Step 1: Assessing the water samples


In [49]:
df_work

,compartment,measpointname,date,cname,val_mgl,uname_mgl,val_mol_l,orchestra_param
2036,BB11N,PP-11N,2023-12-12,Nikkel [Ni],0.015000,mg/l,2.555802e-07,Ni+2.tot
2037,BB11N,PP-11N,2023-12-12,Chloride,370.000000,mg/l,1.043724e-02,Cl-.tot
2038,BB11N,PP-11N,2023-12-12,Silicium [Si],15.900000,mg/l,5.660377e-04,Si.tot
2039,BB11N,PP-11N,2023-12-12,Temperatuur,18.600000,°C,2.917500e+02,T
2040,BB11N,PP-11N,2023-12-12,Calcium [Ca],410.000000,mg/l,1.022954e-02,Ca+2.tot
2041,BB11N,PP-11N,2023-12-12,Magnesium [Mg],120.000000,mg/l,4.936240e-03,Mg+2.tot
2042,BB11N,PP-11N,2023-12-12,Arseen [As],0.029000,mg/l,3.870796e-07,As.tot
2043,BB11N,PP-11N,2023-12-12,Zink [Zn],0.027000,mg/l,4.129703e-07,Zn+2.tot
2044,BB11N,PP-11N,2023-12-12,Sulfide,0.120000,mg/l,3.741815e-06,S-2.tot
2045,BB11N,PP-11N,2023-12-12,pH,7.160000,-,7.160000e+00,pH


## Initialise the problem 
In order solve this problem with pyOrchestra we first initialize our problem using the *chemistry_Travertine.inp* file. 
This file predefines the aqueous chemical system in such a way that we can use the information from the tables in the paper as inputs to the Orchestra simulation.

Once pyOrchestra is initialized, running a simulation consists of a series of steps where the values of the required set of input variables are passed via *InVARS* to ORCHESTRA, after which a set of corresponding output variables are passed back in *OutVars*.

ORCHESTRA is initialized in pyOrchestra using the inputfile *chemistry_Travertine.inp*, created above with the ORCHESTRA-GUI. After initialization in Python, we know which variables will be passed through *OutVars* and can be used in *InVars*.

The following code implements these steps.


In [50]:
# We get the list of primary states from df_work and the corresponding parameter names in 
# the Orchestra input file.

primary_states = df_work['orchestra_param'].tolist()
print(primary_states)

# We can now copy the content of this list to the InVars1 list for the Orchestra interface.

# We obtain the required outputs for the dissolved species using the OrchestraGUI

out_logact_diss = [
    'Ar.logact', 'AsO4-3.logact', 'CO2.logact', 'CO3-2.logact', 'Ca+2.logact', 
    'CaCO3.logact', 'CaHCO3+.logact', 'CaOH+.logact', 'CaSO4.logact', 
    'Ca[HPO4].logact', 'Ca[OH]+.logact', 'Ca[SO4].logact', 'Cl-.logact', 
    'Fe+2.logact', 'FeCO3.logact', 'FeCl+.logact', 'FeCl2.logact', 'FeCl3-.logact', 
    'Fe[CO3]2-2.logact', 'Fe[H2PO4]+.logact', 'Fe[HPO4].logact', 'Fe[HS]+.logact', 
    'Fe[HS]2.logact', 'Fe[NH3]+2.logact', 'Fe[NH3]2+2.logact', 'Fe[NH3]4+2.logact', 
    'Fe[OH]+.logact', 'Fe[OH]2.logact', 'Fe[OH]3-.logact', 'Fe[OH]4-2.logact', 
    'Fe[SO4].logact', 'H+.logact', 'H2CO3.logact', 'H2S.logact', 'H2[AsO4]-.logact', 
    'H2[PO4]-.logact', 'H2[SiO4]-2.logact', 'H3[AsO4].logact', 'H3[PO4].logact', 
    'H3[SiO4]-.logact', 'H4[SiO4].logact', 'HCO3-.logact', 'HPO4-2.logact', 
    'HS-.logact', 'HSO4-.logact', 'H[AsO4]-2.logact', 'K+.logact', 'KPO4-2.logact', 
    'KSO4-.logact', 'K[HPO4]-.logact', 
    'Mg+2.logact', 'MgCO3.logact', 'MgHCO3+.logact', 'MgOH+.logact', 'MgSO4.logact', 
    'Mg[H2PO4]+.logact', 'Mg[H3SiO4]+.logact', 'Mg[HPO4].logact', 'Mg[NH3]+2.logact', 
    'Mg[NH3]2+2.logact', 'Mg[NH3]3+2.logact', 'Mg[NH3]4+2.logact', 'Mg[PO4]-.logact', 
    'Mn+2.logact', 'Mn2[OH]+3.logact', 'Mn2[OH]3+.logact', 'MnCl+.logact', 'MnCl2.logact', 
    'MnCl3-.logact', 'Mn[CO3].logact', 'Mn[HCO3]+.logact', 'Mn[HPO4].logact', 
    'Mn[HPO4]2-2.logact', 'Mn[NH3]+2.logact', 'Mn[NH3]2+2.logact', 'Mn[OH]+.logact', 
    'Mn[OH]2.logact', 'Mn[OH]3-.logact', 'Mn[OH]4-2.logact', 'Mn[SO4].logact', 
    'NH3.logact', 'NH4+.logact', 'Na+.logact', 'NaCO3-.logact', 'NaH2PO4.logact', 
    'NaHCO3.logact', 'NaPO4-2.logact', 'NaSO4-.logact', 'Na[HPO4]-.logact', 
    'Ni+2.logact', 'Ni2[OH]+3.logact', 'Ni4[OH]4+4.logact', 'NiCl+.logact', 
    'NiHAsO4.logact', 'NiHS+.logact', 'Ni[CO3].logact', 'Ni[CO3]2-2.logact', 
    'Ni[HCO3]+.logact', 'Ni[HPO4].logact', 'Ni[HS]2.logact', 'Ni[NH3]+2.logact', 
    'Ni[NH3]2+2.logact', 'Ni[NH3]3+2.logact', 'Ni[NH3]4+2.logact', 'Ni[OH]+.logact', 
    'Ni[OH]2.logact', 'Ni[OH]2[HPO4]-2.logact', 'Ni[SO4].logact', 'Ni[SO4]2-2.logact', 
    'OH-.logact', 'PO4-3.logact', 'S-2.logact', 'SO4-2.logact', 'Si2O2[OH]5-.logact', 
    'Si2O3[OH]4-2.logact', 'Si3O5[OH]5-3.logact', 'Si3O6[OH]3-3.logact', 
    'Si4O6[OH]6-2.logact', 'Si4O7[OH]6-4.logact', 'Si4O8[OH]4-4.logact', 
    'Si6O15-6.logact', 'Zn+2.logact'
]

out_si_minerals = [
    'Anhydrite[s].si', 'Aragonite[s].si', 'Calcite[s].si', 'Dolomite[s].si', 'FeS[ppt][s].si', 
    'Gypsum[s].si', 'Halite[s].si', 'Hydroxyapatite[s].si', 'Mackinawite[s].si', 
    'Melanterite[s].si', 'Pyrochroite[s].si', 'Quartz[s].si', 'Rhodochrosite[s].si', 
    'Siderite[s].si', 'Smithsonite[s].si', 'Sphalerite[s].si', 'Sylvite[s].si', 
    'Talc[s].si', 'Vivianite[s].si', 'Zn[OH]2[e][s].si'
]

out_logact_gases = ['Ar[g].logact', 'CO2[g].logact']

out_extra = ['chargebalance', 'I']

['Ni+2.tot', 'Cl-.tot', 'Si.tot', 'T', 'Ca+2.tot', 'Mg+2.tot', 'As.tot', 'Zn+2.tot', 'S-2.tot', 'pH', 'K+.tot', 'NH4+.tot', 'Mn+2.tot', 'HCO3-.tot', 'Na+.tot', 'PO4-3.tot', 'SO4-2.tot', 'Fe+2.tot']


In [51]:
test = primary_states + out_logact_diss + out_si_minerals + out_logact_gases + out_extra
print(test)
print(np.array(test))

['Ni+2.tot', 'Cl-.tot', 'Si.tot', 'T', 'Ca+2.tot', 'Mg+2.tot', 'As.tot', 'Zn+2.tot', 'S-2.tot', 'pH', 'K+.tot', 'NH4+.tot', 'Mn+2.tot', 'HCO3-.tot', 'Na+.tot', 'PO4-3.tot', 'SO4-2.tot', 'Fe+2.tot', 'Ar.logact', 'AsO4-3.logact', 'CO2.logact', 'CO3-2.logact', 'Ca+2.logact', 'CaCO3.logact', 'CaHCO3+.logact', 'CaOH+.logact', 'CaSO4.logact', 'Ca[HPO4].logact', 'Ca[OH]+.logact', 'Ca[SO4].logact', 'Cl-.logact', 'Fe+2.logact', 'FeCO3.logact', 'FeCl+.logact', 'FeCl2.logact', 'FeCl3-.logact', 'Fe[CO3]2-2.logact', 'Fe[H2PO4]+.logact', 'Fe[HPO4].logact', 'Fe[HS]+.logact', 'Fe[HS]2.logact', 'Fe[NH3]+2.logact', 'Fe[NH3]2+2.logact', 'Fe[NH3]4+2.logact', 'Fe[OH]+.logact', 'Fe[OH]2.logact', 'Fe[OH]3-.logact', 'Fe[OH]4-2.logact', 'Fe[SO4].logact', 'H+.logact', 'H2CO3.logact', 'H2S.logact', 'H2[AsO4]-.logact', 'H2[PO4]-.logact', 'H2[SiO4]-2.logact', 'H3[AsO4].logact', 'H3[PO4].logact', 'H3[SiO4]-.logact', 'H4[SiO4].logact', 'HCO3-.logact', 'HPO4-2.logact', 'HS-.logact', 'HSO4-.logact', 'H[AsO4]-2.logact'

In [52]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_leachate.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars1 = np.array([
        'pH', 'T', 'Ni+2.tot', 'Cl-.tot', 'Si.tot', 'Ca+2.tot', 
        'Mg+2.tot', 'As.tot', 'Zn+2.tot', 'S-2.tot',  
        'K+.tot', 'NH4+.tot', 'Mn+2.tot', 'HCO3-.tot', 
        'Na+.tot', 'PO4-3.tot', 'SO4-2.tot', 'Fe+2.tot',
        'watervolume', 'gasvolume'
    ])
    
    # We select the output from Orchestra we need to use
    # We use the Output selector tab in the GUI to select the output variables.
    out_list = primary_states + out_logact_diss + out_si_minerals + out_logact_gases + out_extra
    OutVars1 = np.array(out_list)
        

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)


Reading and expanding calculator new stylechemistry_leachate.inp
Scanning file: chemistry_leachate.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
Scanning file: chemistry_leachate.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
Including file: chemistry_leachate.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
0.136 sec.
	Reading variables .... 0.086 s
testing:
16:pH
4:T
30:Ni+2.tot
13:Cl-.tot
38:Si.tot
11:Ca+2.tot
22:Mg+2.tot
9:As.tot
40:Zn+2.tot
34:S-2.tot
20:K+.tot
26:NH4+.tot
24:Mn+2.tot
18:HCO3-.tot
28:Na+.tot
32:PO4-3.tot
36:SO4-2.tot
15:Fe+2.tot
41:watervolume
42:gasvolume
30:Ni+2.tot
13:Cl-.tot
38:Si.tot
4:T
11:Ca+2.tot
22:Mg+2.tot
9:As.tot
40:Zn+2.tot
34:S-2.tot
16:pH
20:K+.tot
26:NH4+.tot
24:Mn+2.tot
18:HCO3-.tot
28:Na+.tot
32:PO4-3.tot
36:SO4-2.tot
15:Fe+2.tot
43:Ar.logact
44:AsO4-3.logact
45:CO2.logact
46:CO3-2.logact
10:Ca+2.logact
47:CaCO3.logact
48:CaHCO3+.logact
49:CaOH+.logact
50:CaSO4.logact
51:Ca[HPO4].logact
5

Please note that the output of this code is what ORCHESTRA echos back. ORCHESTRA uses a set of variables in order to store the input variables and the results of the calculations, in this case 33. The top part of the output shows the output requested by us through *OutVars* together with the values used during initialization.

### Run the Problem

1. We use the data imported as the mass of our primary states.
2. We check the output for relevant species. 
    - we exported all possible species. We can negelect all species with very low activities;
    - we exported all possible minerals. We only need to include the ones with relatively high SI-values > -0.5?



In [53]:
# Let us print the initial primary state values

table_md_data = df_work.to_markdown()
display(Markdown(table_md_data))

# print(df_data)

|      | compartment   | measpointname   | date                | cname              |    val_mgl | uname_mgl   |     val_mol_l | orchestra_param   |
|-----:|:--------------|:----------------|:--------------------|:-------------------|-----------:|:------------|--------------:|:------------------|
| 2036 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Nikkel [Ni]        |    0.015   | mg/l        |   2.5558e-07  | Ni+2.tot          |
| 2037 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Chloride           |  370       | mg/l        |   0.0104372   | Cl-.tot           |
| 2038 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Silicium [Si]      |   15.9     | mg/l        |   0.000566038 | Si.tot            |
| 2039 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Temperatuur        |   18.6     | °C          | 291.75        | T                 |
| 2040 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Calcium [Ca]       |  410       | mg/l        |   0.0102295   | Ca+2.tot          |
| 2041 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Magnesium [Mg]     |  120       | mg/l        |   0.00493624  | Mg+2.tot          |
| 2042 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Arseen [As]        |    0.029   | mg/l        |   3.8708e-07  | As.tot            |
| 2043 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Zink [Zn]          |    0.027   | mg/l        |   4.1297e-07  | Zn+2.tot          |
| 2044 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Sulfide            |    0.12    | mg/l        |   3.74181e-06 | S-2.tot           |
| 2045 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | pH                 |    7.16    | -           |   7.16        | pH                |
| 2046 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Kalium [K]         |  200       | mg/l        |   0.00511509  | K+.tot            |
| 2047 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Ammonium (als NH4) |  372.319   | mg/l        |   0.0206385   | NH4+.tot          |
| 2048 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Mangaan [Mn]       |    1       | mg/l        |   1.82017e-05 | Mn+2.tot          |
| 2049 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Bicarbonaat        | 2600       | mg/l        |   0.042609    | HCO3-.tot         |
| 2050 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Natrium [Na]       |  400       | mg/l        |   0.0173989   | Na+.tot           |
| 2051 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Fosfaat (als PO4)  |    7.49261 | mg/l        |   7.88945e-05 | PO4-3.tot         |
| 2052 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | Sulfaat (als SO4)  |  769.626   | mg/l        |   0.00801193  | SO4-2.tot         |
| 2053 | BB11N         | PP-11N          | 2023-12-12 00:00:00 | IJzer [Fe]         |    4.2     | mg/l        |   7.52014e-05 | Fe+2.tot          |

In [54]:
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
for param in df_work['orchestra_param']:
    print(param)
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[df_work['orchestra_param'] == param, 'val_mol_l'].values[0]

print(IN1)

Ni+2.tot
Cl-.tot
Si.tot
T
Ca+2.tot
Mg+2.tot
As.tot
Zn+2.tot
S-2.tot
pH
K+.tot
NH4+.tot
Mn+2.tot
HCO3-.tot
Na+.tot
PO4-3.tot
SO4-2.tot
Fe+2.tot
[[7.16000000e+00 2.91750000e+02 2.55580167e-07 1.04372355e-02
  5.66037736e-04 1.02295409e-02 4.93624023e-03 3.87079552e-07
  4.12970327e-07 3.74181478e-06 5.11508951e-03 2.06385167e-02
  1.82016746e-05 4.26089807e-02 1.73988691e-02 7.88945263e-05
  8.01192694e-03 7.52014324e-05 1.00000000e+00 1.00000000e+00]]


In [55]:
InVars1

array(['pH', 'T', 'Ni+2.tot', 'Cl-.tot', 'Si.tot', 'Ca+2.tot', 'Mg+2.tot',
       'As.tot', 'Zn+2.tot', 'S-2.tot', 'K+.tot', 'NH4+.tot', 'Mn+2.tot',
       'HCO3-.tot', 'Na+.tot', 'PO4-3.tot', 'SO4-2.tot', 'Fe+2.tot',
       'watervolume', 'gasvolume'], dtype='<U11')

In [56]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars1)])

for param in df_work['orchestra_param']:
    print(param)
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[df_work['orchestra_param'] == param, 'val_mol_l'].values[0]

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'gasvolume')] = 0.0 # no gas 

# run ORCHESTRA
OUT = pO1.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation = pd.DataFrame([all_Res],columns=OutVars1, index=['first calculation'])

table_mdini = Res_Simulation[['T','pH', 'HCO3-.logact', 
                      'Ca+2.logact', 'CO2[g].logact', 
                      'Calcite[s].si', 'Gypsum[s].si']].to_markdown()
display(Markdown(table_mdini))


Ni+2.tot
Cl-.tot
Si.tot
T
Ca+2.tot
Mg+2.tot
As.tot
Zn+2.tot
S-2.tot
pH
K+.tot
NH4+.tot
Mn+2.tot
HCO3-.tot
Na+.tot
PO4-3.tot
SO4-2.tot
Fe+2.tot
Try a first calculation with iia switched off....
Parsing expressions of chemistry_leachate.inp..... 
Optimizing expressions of chemistry_leachate.inp..... 0.509 sec.
8370 variables, 24773 expressions, 16 equations.
First calculation was successful!
Repeat calculation with iia switched on..
Switching on: logI: -2
Switching on: pH: 7.16
This was successful!!


|                   |      T |      pH |   HCO3-.logact |   Ca+2.logact |   CO2[g].logact |   Calcite[s].si |   Gypsum[s].si |
|:------------------|-------:|--------:|---------------:|--------------:|----------------:|----------------:|---------------:|
| first calculation | 291.75 | 8.52709 |       -1.52424 |      -2.64191 |        -2.27385 |         2.42508 |      -0.699382 |

In [34]:
print(
    OutVars1.shape,
    all_Res.shape
)

pd.DataFrame([all_Res],columns=OutVars1, index=['first calculation'])

(298,) (298,)


,Ni+2.tot,Cl-.tot,Si.tot,T,Ca+2.tot,Mg+2.tot,As.tot,Zn+2.tot,S-2.tot,pH,...,Trona[s].si,Truscottite[s].si,Vaterite[s].si,Vivianite[s].si,Xonotlite[s].si,Zn3[AsO4]2[s].si,Ar[g].logact,CO2[g].logact,chargebalance,I
first calculation,2.555802e-07,0.010437,0.000566,291.75,0.01023,0.004936,3.870796e-07,4.129703e-07,0.000004,7.310146,...,-11.467594,-48.466393,-0.606631,1.501339,-51.064167,-15.199396,0.0,-1.188245,0.0,0.062152


## Interpretation of the results
The results clearly show that the water samples are not in equilibrium with the atmosphere and calcite. The partial pressure of $\text{CO}_2[g]$ is about 400 ppm which would be 0.0004 at which is lower that calculated above. In addition the SI for calcite is above zero.

What is also striking to see is that the SI for calcite increases along the stream, the CO2[g].logact decreases along the stream and we know that Calcite is precipitating from the stream. How much calcite will have precipitated moving from S-1 to F-6?

## Step 2: Calculate amount of calcite precipitated along stream from S-1 to F-6
In order to run this scenario, we need a different type of calculation than used for step 1. The main difference is that we need to allow Orchestra to precipitate calcite. We need to activate the precipitation reactions in the GUI, which allows Orchestra to add the reactions to the equation set. In addition we need allow our Python script to fix the CO2[g].logact value, instead of defining the HCO3-.tot amount.

```{exercise} Explain
Why do we need to fix the CO2[g].logact value instead of defining the HCO3-.tot amount?
```
In addition to allowing Calcite to precipitate, we also want to be able to adjust the SI value to control the equilibrium condition of Calcite. We can achieve this by adding a constant to the precipitation reaction of Calcite in the *chemistry_Travertine.inp* file. In this case we added *deltaSIcalcite* to the file, with a default value of 0. If we set to another value, the model solves an equilibrium for this other value. These changes have been implemented in the *chemistry_Travertine_fixed_CO2_logact.inp* file. We will allow Orchestra to change the pH in order to achieve charge balance.

We can now initialise a second Orchestra calculator for this input file.


In [ ]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry2.inp' file.
# Later we use pO2, InVars2 and OutVars2
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_Travertine_fixed_CO2_logact.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars2 = np.array(['T','CO2[g].logact', 'Ca+2.tot',  'Mg+2.tot',
                       'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot',
                       'deltaSIcalcite',
                       'watervolume', 'gasvolume'
                       ])
    
    # We select the output from Orchestra we need to use
    OutVars2 = np.array(['T','pH', 
                        'HCO3-.tot', 'HCO3-.con', 'HCO3-.logact', 'HCO3-.diss',
                        'H2CO3.tot', 'H2CO3.con', 'H2CO3.logact', 'H2CO3.diss',
                        'CO3-2.tot', 'CO3-2.con', 'CO3-2.logact', 'CO3-2.diss',
                        'CO2[g].tot', 'CO2[g].con', 'CO2[g].logact', 'CO2[g].diss',
                        'SO4-2.tot', 'SO4-2.con', 'SO4-2.logact','SO4-2.diss',
                        'HSO4-.tot', 'HSO4-.con', 'HSO4-.logact',
                        'Ca+2.tot', 'Ca+2.con', 'Ca+2.logact', 'Ca+2.diss', 
                        'CaF+.con', 'CaOH+.con', 'CaSO4.con',
                        'Calcite[s].si','Calcite[s].tot','Calcite[s].logact',
                        'Fluorite[s].si','Fluorite[s].tot','Fluorite[s].logact',
                        'Gypsum[s].si', 'Gypsum[s].tot','Gypsum[s].logact',
                        'I', 'chargebalance', 'watervolume' ])

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO2 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO2.initialise(InputFile, NoCells, InVars2, OutVars2)

## Strategy to solve the problem
After initializing the Orchestra Calculator, we can solve the problem using our standard approach:
### 1. Define the initial conditions
The initial condition is determined by the measured (total) concentrations in the water sample at S-1.
### 2. Define the boundary conditions
This is the crucial step for the scenario. The sample at F-6 clearly is in a different condition that S-1. The SI value for calcite is different, the CO2-pressure (CO2[g].con, or CO2[g].logact) is different and finally so is the temperature.

We can set these conditions using the InVars2 array, where we use the masses from S-1 and  T, CO2[g].logact and SI Calcite from F-6 all estimated in the previous calculation. The SI-Calcite calculated for F-6 is used for the deltaSIcalcite variable.

We only carry out one simulation.

In [ ]:
# Step 1, initial condition.
# InVars need to contain floats
IN2 = np.array([np.ones_like(InVars2)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars2)])

# set default watervolume and gasvolume
IN2[0][np.where(InVars2 == 'watervolume')] = 1.0 # per liter
IN2[0][np.where(InVars2 == 'gasvolume')] = 0.0 # no gas 

# Set initial concentration values (from point S-1)
inistates = ['Ca+2.tot',  'Mg+2.tot',
             'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']

IN2[0][np.where(np.isin(InVars2,inistates))] = df_data.loc['S-1',inistates].values

# We obtain the target boundary values from Res_Simulation
bndvals = ['T','CO2[g].logact']

IN2[0][np.where(np.isin(InVars2,bndvals))] = Res_Simulation.loc['F-6',bndvals].values
IN2[0][np.where(np.isin(InVars2,'deltaSIcalcite'))] = Res_Simulation.loc['F-6','Calcite[s].si']

# Calculate equilibrium conditions for initial situation (Point A in chapter 4.1)
OUT = pO2.set_and_calculate(IN2)
Res_Sim2 = pd.DataFrame(OUT,columns=OutVars2)

# Need to add the deltaSIcalcite value to the Calcite[s].si value in the output (Orchestra does not do this automatically)
Res_Sim2['Calcite[s].si'] += Res_Simulation.loc['F-6','Calcite[s].si']

Res_Sim2.rename(index={0: 'S-1 to F-6'}, inplace=True)

print("Equilibrate sample from S-1 to F-6")
# print(Res_Sim2[['T','pH','Ca+2.con', 'HCO3-.con', 'CO2[g].con', 'Calcite[s].si', 'Calcite[s].tot', 'CO2[g].logact']])
table_md1 = Res_Sim2[['T','pH','Ca+2.con', 'HCO3-.con', 'CO2[g].con', 'Calcite[s].si', 'Calcite[s].tot', 'CO2[g].logact']].to_markdown()
display(Markdown(table_md1))


print("Initial conditions")

display(Markdown(table_mdini))

## Analysis
In the simulation above we assume that we can obtain a water sample similar to F-6 from S-1 by changing the SI of calcite and ensuring the temperature and the partial pressure $\text{CO}_2\text[g]$ similar to those of F-6. Clearly this is not completely true which becomes clear when we compare the pH and and the biocarbonate concentrations. 

**Selfstudy questions:** 
+ What could be the reason for these differences? 
+ What was the explanation in the paper?
+ How would you change the above code in order to see the changes for the other species present in this system?

## Assignment
In the above we have analysed two of the samples shown in Lorah and Herman (1988). Your task is to analyse all other results in a similar way. For this assignment you need to do the following:

1. Import all data from Table 1 in Lorah and Herman.
2. Translate all values to K and mg/l where relevant.
3. Add additional information from Lorah and Herman so you can make nice plots. For example, add the distance along the stream which you can estimate from the map shown in Figure 1.
4. Create plots of the raw data similar to figures 2 and 3 to better understand the data.
5. Evaluate all measurements using Orchestra with the *chemistry_Travertine.inp* file.
6. Carefully evaluate the results, make plots for the CO2[g].con, CO2[g].logact, Calcite[s].si etc. so that you understand how the chemistry changes along the stream.
7. Set up a series of Orchestra calculations using the *chemistry_Travertine_fixed_CO2_logact.inp* file in order to estimate the changes in the amounts of HCO3-, Calcite and Ca+2 along the stream.
8. Finally, use the Orchestra to estimate the total amount of Calcite that can precipitate from the samples when they would be brought to equilibrium with the atmosphere in the laboratory at 25 $^\text{o} \text{C}$.


# Analysis of all data from Lorah & Herman

In [ ]:
# copy data from Lorah and Herman (1988)
# We store the data in a data frame
# Because the data are given in mg/L we need the molar masses to be able to 
# translate to mol/liter

# for conveniece we set mmass for non chemical values to 0.001 because it is then very easy
# to calculate the concentrations in mol per liter...
# Please note: we use Orchestra notation

# Data October 14, 1984

headers = ['Sample','T_celsius', 'Conductivity', 'pH', 'HCO3-.tot', 'Ca+2.tot',  
           'Mg+2.tot', 'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']
data = [
    ['mmass', 1e-3, 1e-3, 1e-3, 61.02, 40.08, 24.31, 23, 39.1, 19, 34.45, 96.07],
    ['S-1', 24.3, 700, 7.32, 308, 165, 25.4, 5.1, 14.2, 1, 3.3, 283],
    ['S-3', 24.6, 730, 7.23, 316,168,28,3.9,14.8,1.1,3.5,291],
    ['S-2', 24.7, 720, 7.2, 315,169,26.2,3.9,14.3,1.1,5.8,296],
    ['D-1', 24.3, 730, 7.24, 310,168,26,3.4,14.2,1.1,3.2,283],
    ['D-3', 23.6, np.nan, 7.43, 314, 168, 26.2, 2.8, 12.9, 1.2, 3.5, 288],
    ['F-1', 21.4, 695, 7.79, 312,166,25.6,3.5,13.9,1.2,3.4,286],
    ['F-3', 20.5, 695, 8.06, 311,164,25.2,3.7,14,1,3.5,283],
    ['F-4', 19.7, 690, 8.27, 278,160,25.6,3.6,14.3,1,3.8,286],
    ['F-5', 19.6, 690, 8.28, 273,157,25.5,3.1,14,1.1,3.4,283],
    ['F-2', 18.4, 660, 8.37, 264,151,25.6,3.2,14.3,1,3.4,283],
    ['F-6', 14.4, 442, 8.25, 210,122,21.3,3.3,11.4,0.7,4,246]
    ]


df_data_orig_1984 = pd.DataFrame(data=data,columns=headers)
df_data_orig_1984.set_index('Sample',inplace=True)

# calculate molar concentrations
#df_data_1984 = pd.DataFrame()
df_data_1984 = df_data_orig_1984.iloc[1:]/df_data_orig_1984.loc['mmass'] * 0.001
# calculate temperature in K
df_data_1984['T'] = df_data_1984['T_celsius'] + 273.15

table_md_data = df_data_1984.to_markdown()
display(Markdown(table_md_data))

# print(df_data)

In [ ]:
# copy data from Lorah and Herman (1988)
# We store the data in a data frame
# Because the data are given in mg/L we need the molar masses to be able to 
# translate to mol/liter

# for conveniece we set mmass for non chemical values to 0.001 because it is then very easy
# to calculate the concentrations in mol per liter...
# Please note: we use Orchestra notation

# Data April 6, 1985

headers = ['Sample','T_celsius', 'Conductivity', 'pH', 'HCO3-.tot', 'Ca+2.tot',  
           'Mg+2.tot', 'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']
data = [
    ['mmass', 1e-3, 1e-3, 1e-3, 61.02, 40.08, 24.31, 23, 39.1, 19, 34.45, 96.07],
    ['S-1' ,20,590,6.98,234,102,17,2.5,6.8,0.5,3.3,150,],
    ['S-3',20,600,6.91,239,105,17.1,2.9,7.3,0.6,3.9,159,],
    ['S-2',20,600,6.89,240,106,16.6,2.4,6.9,0.6,3.8,159,],
    ['D-1',19.5,600,7.01,237,103,17.2,3,7.1,0.6,3.6,158,],
    ['D-3',17,595,7.19,237,104,16.6,2.4,6.6,0.6,3.8,158,],
    ['F-1',17,600,7.41,235,104,16.9,2.5,6.5,0.6,3.7,160,],
    ['F-3',16.5,520,7.83,234,102,16.5,2.8,6.9,0.6,3.8,162,],
    ['F-4',13,510,7.98,224,100,16.3,2.7,6.6,0.6,3.7,162,],
    ['F-5',13.5,488,7.92,220,99,16.2,2.5,6.7,0.6,3.8,162,],
    ['F-2',13.5,461,7.98,209,97,16.7,2.3,6.4,0.6,3.9,164,],
    ['F-6',11,393,8.08,183,83,15,2.5,5.8,0.6,4.3,142,],
]
    

df_data_orig_1985 = pd.DataFrame(data=data,columns=headers)
df_data_orig_1985.set_index('Sample',inplace=True)

# calculate molar concentrations
#df_data_1985 = pd.DataFrame()
df_data_1985 = df_data_orig_1985.iloc[1:]/df_data_orig_1985.loc['mmass'] * 0.001
# calculate temperature in K
df_data_1985['T'] = df_data_1985['T_celsius'] + 273.15

table_md_data = df_data_1985.to_markdown()
display(Markdown(table_md_data))

# print(df_data)

In [ ]:
# %%
# All 1984 data
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([len(df_data_1984),len(OutVars1)])

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'gasvolume')] = 0.0 # no gas 

# loop over available samplesCa_Range 
# we use a counter ii in order to store the results
for ii in range(len(df_data_1984)):
    # set InVars
    # please note InVars was initialized assuming the same frequency as
    # Lorah and Herman. 

    # please note, we fill until -2, watervolume and gasvolume have already
    # been defined above. Please note we use "Orchestra" notation
    IN1[0][:-2] = df_data_1984.iloc[ii][['T', 'pH', 'HCO3-.tot', 'Ca+2.tot', 'Mg+2.tot', 
        'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']].values
    
    # run ORCHESTRA
    OUT = pO1.set_and_calculate(IN1)
    all_Res[ii] = OUT[0]

    # Create a dataframe from all_Res
Res_Sample_Analysis_1984 = pd.DataFrame(all_Res,columns=OutVars1, index=df_data_1984.index)

table_mdini = Res_Sample_Analysis_1984[['T','pH', 'HCO3-.con', 
                      'Ca+2.con', 'CO2[g].con','CO2[g].logact', 
                      'Calcite[s].si', 'Gypsum[s].si', 'Fluorite[s].si']].to_markdown()
display(Markdown(table_mdini))


In [ ]:
# %%
# All 1985 data
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([len(df_data_1985),len(OutVars1)])

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'gasvolume')] = 0.0 # no gas 

# loop over available samplesCa_Range 
# we use a counter ii in order to store the results
for ii in range(len(df_data_1985)):
    # set InVars
    # please note InVars was initialized assuming the same frequency as
    # Lorah and Herman. 

    # please note, we fill until -2, watervolume and gasvolume have already
    # been defined above. Please note we use "Orchestra" notation
    IN1[0][:-2] = df_data_1985.iloc[ii][['T', 'pH', 'HCO3-.tot', 'Ca+2.tot', 'Mg+2.tot', 
        'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']].values
    
    # run ORCHESTRA
    OUT = pO1.set_and_calculate(IN1)
    all_Res[ii] = OUT[0]

    # Create a dataframe from all_Res
Res_Sample_Analysis_1985 = pd.DataFrame(all_Res,columns=OutVars1, index=df_data_1985.index)

table_mdini = Res_Sample_Analysis_1985[['T','pH', 'HCO3-.con', 
                      'Ca+2.con', 'CO2[g].con','CO2[g].logact', 
                      'Calcite[s].si', 'Gypsum[s].si', 'Fluorite[s].si']].to_markdown()
display(Markdown(table_mdini))


In [ ]:
df_data_1985.iloc[ii]